# Beacon WebSocket SQL Test Client

This notebook tests `/api/ws/sql` with either Basic auth handshake or ticket-based auth.

## Requirements

- `BEACON_WS_SQL_ENABLE=true`
- `BEACON_ENABLE_SQL=true`
- Python packages: `websockets`, `httpx`, `pyarrow`


In [1]:
# Optional: install dependencies in the active kernel
%pip install websockets httpx pyarrow


   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   ---------------- ----------------------- 2/5 [anyio]
   ---------------- ----------------------- 2/5 [anyio]
   ------------------------ --------------- 3/5 [httpcore]
   -------------------------------- ------- 4/5 [httpx]
   -------------------------------- ------- 4/5 [httpx]
   ---------------------------------------- 5/5 [httpx]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import asyncio
import base64
import json
import uuid

import httpx
import pandas as pd
import pyarrow.ipc as ipc
import websockets

In [3]:
# Configuration
HOST = "127.0.0.1"
PORT = 5001
USE_TLS = False
AUTH_MODE = "basic"  # "ticket" or "basic"
USERNAME = "beacon-admin"
PASSWORD = "beacon-password"
SQL = "SELECT 1"
TIMEOUT_SECS = 30.0
WS_PING_INTERVAL_SECS = 20.0
WS_PING_TIMEOUT_SECS = None  # None disables client-side ping timeout
WS_MAX_SIZE_BYTES = None  # None disables the 1 MiB default message limit in websockets

In [4]:
def build_basic_auth_header(username: str, password: str) -> dict:
    token = base64.b64encode(f"{username}:{password}".encode("utf-8")).decode("ascii")
    return {"Authorization": f"Basic {token}"}


async def fetch_ws_ticket(http_base: str, username: str, password: str, timeout: float) -> str:
    headers = build_basic_auth_header(username, password)
    async with httpx.AsyncClient(timeout=timeout) as client:
        response = await client.post(f"{http_base}/api/ws/sql-ticket", headers=headers)
        response.raise_for_status()
        payload = response.json()
        return payload["ticket"]


def decode_arrow_stream_to_dataframes(binary_payload: bytes):
    row_count = 0
    dataframes = []
    reader = ipc.open_stream(binary_payload)
    for batch in reader:
        row_count += batch.num_rows
        dataframes.append(batch.to_pandas())
    return row_count, dataframes


async def resolve_ws_endpoint(
    auth_mode: str | None = None,
    timeout: float | None = None,
    use_tls: bool | None = None,
    host: str | None = None,
    port: int | None = None,
    username: str | None = None,
    password: str | None = None,
):
    auth_mode = auth_mode or AUTH_MODE
    timeout = timeout or TIMEOUT_SECS
    use_tls = USE_TLS if use_tls is None else use_tls
    host = host or HOST
    port = port or PORT
    username = username or USERNAME
    password = password or PASSWORD

    scheme_http = "https" if use_tls else "http"
    scheme_ws = "wss" if use_tls else "ws"

    http_base = f"{scheme_http}://{host}:{port}"
    ws_base = f"{scheme_ws}://{host}:{port}/api/ws/sql"

    if auth_mode == "basic":
        headers = build_basic_auth_header(username, password)
        ws_url = ws_base
    elif auth_mode == "ticket":
        ticket = await fetch_ws_ticket(http_base, username, password, timeout)
        headers = None
        ws_url = f"{ws_base}?ticket={ticket}"
    else:
        raise ValueError("AUTH_MODE must be 'basic' or 'ticket'")

    return ws_url, headers, timeout


async def run_sql_dataframe(
    sql: str,
    *,
    auth_mode: str | None = None,
    timeout: float | None = None,
    ping_interval: float | None = None,
    ping_timeout: float | None = None,
    max_size: int | None = None,
    verbose: bool = True,
):
    ws_url, headers, timeout = await resolve_ws_endpoint(auth_mode=auth_mode, timeout=timeout)
    ping_interval = WS_PING_INTERVAL_SECS if ping_interval is None else ping_interval
    ping_timeout = WS_PING_TIMEOUT_SECS if ping_timeout is None else ping_timeout
    max_size = WS_MAX_SIZE_BYTES if max_size is None else max_size
    request_id = str(uuid.uuid4())
    total_rows = 0
    dataframes = []

    if verbose:
        print(f"Connecting: {ws_url}")

    async with websockets.connect(
        ws_url,
        additional_headers=headers,
        open_timeout=timeout,
        ping_interval=ping_interval,
        ping_timeout=ping_timeout,
        max_size=max_size,
    ) as ws:
        await ws.send(json.dumps({
            "type": "run_sql",
            "sql": sql,
            "request_id": request_id,
        }))

        if verbose:
            print(f"Sent run_sql request_id={request_id}")

        while True:
            frame = await ws.recv()

            if isinstance(frame, str):
                event = json.loads(frame)
                event_type = event.get("type")

                if verbose:
                    print("text event:", event)

                if event_type == "error":
                    raise RuntimeError(event.get("message", "unknown websocket error"))
                if event_type == "done":
                    break
            else:
                batch_rows, batch_frames = decode_arrow_stream_to_dataframes(bytes(frame))
                total_rows += batch_rows
                dataframes.extend(batch_frames)

    if verbose:
        print(f"Decoded rows from binary stream: {total_rows}")

    if not dataframes:
        return pd.DataFrame()

    return pd.concat(dataframes, ignore_index=True)


async def run_sql(sql: str, **kwargs):
    """Convenience alias for notebook cells: returns a pandas DataFrame."""
    return await run_sql_dataframe(sql, **kwargs)

In [106]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql(SQL)
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=a6c69921-57bf-4725-bd7e-e38880254578
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': 'a6c69921-57bf-4725-bd7e-e38880254578'}
text event: {'type': 'chunk', 'request_id': 'a6c69921-57bf-4725-bd7e-e38880254578', 'batch_index': 0, 'row_count': 1}
text event: {'type': 'done', 'request_id': 'a6c69921-57bf-4725-bd7e-e38880254578', 'total_rows': 1}
Decoded rows from binary stream: 1


,Int64(1)
0,1


In [27]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql("CREATE ATLAS TABLE example LOCATION 'argo-atlas'")
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=415d2eaa-85bf-4d0c-8f99-96488202aa5c
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': '415d2eaa-85bf-4d0c-8f99-96488202aa5c'}
text event: {'type': 'done', 'request_id': '415d2eaa-85bf-4d0c-8f99-96488202aa5c', 'total_rows': 0}
Decoded rows from binary stream: 0


""


In [6]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql("DESCRIBE example")
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=3da9bf73-73bd-42d0-bd81-ed0557a093b5
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': '3da9bf73-73bd-42d0-bd81-ed0557a093b5'}
text event: {'type': 'chunk', 'request_id': '3da9bf73-73bd-42d0-bd81-ed0557a093b5', 'batch_index': 0, 'row_count': 0}
text event: {'type': 'done', 'request_id': '3da9bf73-73bd-42d0-bd81-ed0557a093b5', 'total_rows': 0}
Decoded rows from binary stream: 0


,column_name,data_type,is_nullable


In [28]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql("INGEST INTO ATLAS example ON PARTITION 'base' FROM 'argo-floats/**/*.nc' WITH netcdf")
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=85380b5a-8736-4987-a6c3-70e3fb8ea49c
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': '85380b5a-8736-4987-a6c3-70e3fb8ea49c'}
text event: {'type': 'chunk', 'request_id': '85380b5a-8736-4987-a6c3-70e3fb8ea49c', 'batch_index': 0, 'row_count': 1}
text event: {'type': 'chunk', 'request_id': '85380b5a-8736-4987-a6c3-70e3fb8ea49c', 'batch_index': 1, 'row_count': 1}
text event: {'type': 'chunk', 'request_id': '85380b5a-8736-4987-a6c3-70e3fb8ea49c', 'batch_index': 2, 'row_count': 1}
text event: {'type': 'chunk', 'request_id': '85380b5a-8736-4987-a6c3-70e3fb8ea49c', 'batch_index': 3, 'row_count': 1}
text event: {'type': 'chunk', 'request_id': '85380b5a-8736-4987-a6c3-70e3fb8ea49c', 'batch_index': 4, 'row_count': 1}
text event: {'type': 'chunk', 'request_id': '85380b5a-8736-4987-a6c3-70e3fb8ea49c', 'batch_index': 5, 'row_count': 1}
text event: {'type': 'chunk', 'request_id': '85380b5a-8736-4987-a6c3-70

,partition_name,dataset_name,status,message
0,base,\\?\C:\Git\beacon\data\datasets\argo-floats\pu...,ok,inserted
1,base,\\?\C:\Git\beacon\data\datasets\argo-floats\pu...,ok,inserted
2,base,\\?\C:\Git\beacon\data\datasets\argo-floats\pu...,ok,inserted
3,base,\\?\C:\Git\beacon\data\datasets\argo-floats\pu...,ok,inserted
4,base,\\?\C:\Git\beacon\data\datasets\argo-floats\pu...,ok,inserted
...,...,...,...,...
20618,base,\\?\C:\Git\beacon\data\datasets\argo-floats\pu...,ok,inserted
20619,base,\\?\C:\Git\beacon\data\datasets\argo-floats\pu...,ok,inserted
20620,base,\\?\C:\Git\beacon\data\datasets\argo-floats\pu...,ok,inserted
20621,base,\\?\C:\Git\beacon\data\datasets\argo-floats\pu...,ok,inserted


In [ ]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql("DROP TABLE example")
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=64f6f45e-8dc3-4b39-9e34-9977c7636594
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': '64f6f45e-8dc3-4b39-9e34-9977c7636594'}
text event: {'type': 'done', 'request_id': '64f6f45e-8dc3-4b39-9e34-9977c7636594', 'total_rows': 0}
Decoded rows from binary stream: 0


""


In [33]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql("SELECT JULD, PRES, LONGITUDE, LATITUDE, TEMP FROM example WHERE PRES < 10 AND TEMP IS NOT NULL AND JULD > '2014-01-01T00:00:00' AND JULD < '2014-12-31T23:59:59'")
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=7000796d-1ed1-4536-ae61-873299477517
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': '7000796d-1ed1-4536-ae61-873299477517'}
text event: {'type': 'chunk', 'request_id': '7000796d-1ed1-4536-ae61-873299477517', 'batch_index': 0, 'row_count': 73602}
text event: {'type': 'chunk', 'request_id': '7000796d-1ed1-4536-ae61-873299477517', 'batch_index': 1, 'row_count': 83884}
text event: {'type': 'chunk', 'request_id': '7000796d-1ed1-4536-ae61-873299477517', 'batch_index': 2, 'row_count': 78537}
text event: {'type': 'chunk', 'request_id': '7000796d-1ed1-4536-ae61-873299477517', 'batch_index': 3, 'row_count': 107020}
text event: {'type': 'chunk', 'request_id': '7000796d-1ed1-4536-ae61-873299477517', 'batch_index': 4, 'row_count': 65554}
text event: {'type': 'chunk', 'request_id': '7000796d-1ed1-4536-ae61-873299477517', 'batch_index': 5, 'row_count': 82759}
text event: {'type': 'chunk', 'request_id': '7

,JULD,PRES,LONGITUDE,LATITUDE,TEMP
0,2014-01-02 17:49:59.999999744,1.04,133.191490,9.821850,28.952000
1,2014-01-02 17:49:59.999999744,1.96,133.191490,9.821850,28.957001
2,2014-01-02 17:49:59.999999744,2.96,133.191490,9.821850,28.952999
3,2014-01-02 17:49:59.999999744,4.00,133.191490,9.821850,28.964001
4,2014-01-02 17:49:59.999999744,5.00,133.191490,9.821850,28.964001
...,...,...,...,...,...
923925,2014-12-31 09:54:59.999999744,5.00,-131.080002,49.549999,10.303000
923926,2014-12-31 09:54:59.999999744,6.00,-131.080002,49.549999,10.302000
923927,2014-12-31 09:54:59.999999744,7.00,-131.080002,49.549999,10.302000
923928,2014-12-31 09:54:59.999999744,8.00,-131.080002,49.549999,10.300000
